# Mở rộng băng thông (Bandwidth Extension - BWE)

> Các phần cứng và phần mềm khác nhau sử dụng nhiều tần số lấy mẫu khác nhau, dẫn đến việc phía tái tạo âm thanh thường có thể cung cấp tần số lấy mẫu cao hơn tín hiệu ghi được. Phần cứng cục bộ do đó tốt hơn âm thanh nhận được. Băng thông của âm thanh đầu ra vì thế tương ứng với âm thanh ghi được, có chất lượng thấp hơn những gì phần cứng có thể tái tạo. Sự giới hạn băng thông này làm cho âm thanh đầu ra trở nên đục và tối hơn so với bản gốc (xem [Dạng sóng/Tần số lấy mẫu](content:samplingrate) để nghe các ví dụ âm thanh).
 Mục tiêu ở đây là khắc phục những suy hao chất lượng âm thanh như vậy, khôi phục âm thanh về chất lượng tốt nhất có thể.

**Mở rộng băng thông (Bandwidth Extension - BWE)**, hay đôi khi còn được gọi là **siêu phân giải âm thanh (audio super resolution)** {cite:p}`kuleshov2017audiosuperres` (tương tự như khái niệm siêu phân giải trong xử lý hình ảnh), là tác vụ ánh xạ một tín hiệu $s_l$ có tần số lấy mẫu $fs_l$ sang một tín hiệu khác $s_h$ có tần số lấy mẫu $fs_h$. Thông thường, $fs_h$ lớn hơn $fs_l$ một hệ số $r$ lần, trong đó $r$ là một số nguyên và được gọi là **hệ số giảm mẫu (downsampling factor)**:

$$\frac{fs_h}{fs_l} = r$$

Trong BWE, tín hiệu ước lượng $\hat{s}_h$ của $s_h$ về mặt cảm nhận lý tưởng phải giống như được số hóa ban đầu ở tần số $fs_h$, tức là càng gần với $s_h$ càng tốt. Nếu không biết thêm thông tin nào ngoài $s_l$, bài toán này còn được gọi trong tài liệu học thuật là **mở rộng băng thông mù (Blind Bandwidth Extension - BBWE)** {cite:p}`schmidt2018bbwe`. Nó có thể được phát biểu như sau:

$$\hat{s}_h=f\left(W, s_l\right)$$

Trong đó $W$ là một tập hợp các tham số tùy ý. Cần lưu ý rằng mối quan hệ trên giả định rằng $\hat{s}_h$ có thể được ước lượng từ $s_l$. Điều này có thể không khả thi đối với mọi loại tín hiệu, và vì lý do này, BWE được coi là một bài toán thiếu điều kiện xác định (overdetermined / ill-posed problem) vì nhiều ứng viên $\hat{s}_h$ khác nhau có thể được tạo ra từ cùng một tín hiệu $s_l$. Đối với một trường hợp cụ thể (ví dụ: tín hiệu tiếng nói), một số ứng viên sẽ phù hợp hơn về mặt cảm nhận so với những ứng viên khác; do đó, các tiêu chí cảm nhận đóng vai trò cực kỳ quan trọng trong việc lựa chọn ước lượng phù hợp. Hơn nữa, nhiều câu hỏi vẫn còn bỏ ngỏ như: *Tập tham số $W$ nên chứa bao nhiêu tham số?* hay *Cần bao nhiêu mẫu lân cận của một mẫu cho trước để tạo ra ước lượng tối ưu $\hat{s}_h$?* Các phương pháp được sử dụng rộng rãi trong xử lý tiếng nói như [dự báo tuyến tính](content:linearprediction) khai thác thực tế là các mẫu tín hiệu tiếng nói liền kề có mối tương quan cao và ở tần số lấy mẫu đủ lớn, chúng không thay đổi đột ngột. Yếu tố này rất có lợi cho bài toán BWE tiếng nói, vì nó góp phần hình thành một tiêu chí giúp phân loại các ứng viên phù hợp nhất ra khỏi các ứng viên kém phù hợp hơn.

Các đồ thị dưới đây so sánh một bản ghi âm được lấy mẫu ở tần số 48 kHz và chính bản ghi âm đó nhưng đã được giảm mẫu xuống 16 kHz trước đó.

In [ ]:
import IPython
import librosa
import librosa.display
import matplotlib.pyplot as plt


# Read audio files
audio_48k, _ = librosa.load("sounds/sample-speech-48k.wav", sr=48000)
audio_16k, _ = librosa.load("sounds/sample-speech-16k.wav", sr=16000)

# Display waveform of the speech sample recorded at 48kHz
librosa.display.waveshow(audio_48k, sr=48000)
plt.title("Speech sample at 48kHz")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()
IPython.display.display(IPython.display.Audio(audio_48k, rate=48000))

# Do the same for the file recorded at 16kHz
librosa.display.waveshow(audio_16k, sr=16000)
plt.title("Speech sample at 16kHz")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()
IPython.display.display(IPython.display.Audio(audio_16k, rate=16000))

Có thể thấy rõ rằng mặc dù dạng sóng của cả hai ví dụ âm thanh rất giống nhau, nhưng âm thanh cảm nhận được lại khác biệt do phần thông tin bị mất trong bản ghi âm 16 kHz. Dù thông điệp lời nói vẫn có thể hiểu được trong cả hai trường hợp, nhưng tín hiệu 16 kHz bị thiếu hụt các thành phần tần số cao, ảnh hưởng chủ yếu đến các âm tắc ([plosives](https://en.wikipedia.org/wiki/Plosive)) và âm xát ([fricatives](https://en.wikipedia.org/wiki/Fricative)).

Bây giờ, hãy so sánh phổ biên độ (magnitude spectrogram) của cả hai bản ghi:

In [ ]:
import numpy as np

# The 16k audio will be resampled to match the shape of the 48k audio
audio_16k_resampled = librosa.resample(audio_16k, orig_sr=16000, target_sr=48000)

# Calculate magnitude spectrogram of both audios
mag_spectrogram_16k = librosa.amplitude_to_db(np.abs(librosa.stft(audio_16k_resampled)), ref=np.max)
mag_spectrogram_48k = librosa.amplitude_to_db(np.abs(librosa.stft(audio_48k)), ref=np.max)

# Plot magnitude spectrogram of both audios
librosa.display.specshow(mag_spectrogram_48k, y_axis="log", x_axis="time", sr=48000, cmap="viridis")
plt.title("Magnitude spectrogram of speech sample at 48kHz")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.show()

librosa.display.specshow(mag_spectrogram_16k, y_axis="log", x_axis="time", sr=48000, cmap="viridis")
plt.title("Magnitude spectrogram of speech sample at 16kHz")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.show()

Những gì vừa được trải nghiệm qua việc lắng nghe các mẫu âm thanh đã được xác thực bằng cách so sánh các đồ thị phổ biên độ. Âm thanh ở tần số 16 kHz hoàn toàn thiếu nội dung phía trên một nửa tần số lấy mẫu của nó (tức là trên 8 kHz), và đây chính là phần mà thuật toán BWE sẽ cố gắng tái tạo.

# Ứng dụng

Các bản ghi âm phòng thu chuyên nghiệp cho âm nhạc, phim ảnh hoặc podcast thường sử dụng tần số lấy mẫu tối thiểu là 44,1 kHz (chất lượng CD). Tuy nhiên, các ứng dụng khác như điện thoại VoIP hướng tới tần số lấy mẫu thấp hơn để giảm thiểu lượng dữ liệu cần truyền tải. Ví dụ, tai nghe Bluetooth sử dụng Cấu hình rảnh tay ([Hands-Free Profile - HFP](https://en.wikipedia.org/wiki/List_of_Bluetooth_profiles#Hands-Free_Profile_(HFP))) hoặc Cấu hình tai nghe ([Headset Profile - HSP](https://en.wikipedia.org/wiki/List_of_Bluetooth_profiles#Headset_Profile_(HSP))) sử dụng tần số lấy mẫu 8 kHz (được gọi là **băng hẹp - narrow band**) hoặc 16 kHz (được gọi là **băng rộng - wide band**). Vì lý do này, một trong những động lực chính đằng sau BWE là khắc phục các giới hạn nêu trên nhằm mang lại chất lượng âm thanh tương đương mà không làm ảnh hưởng đến tốc độ truyền dữ liệu. Việc thành công trong mục tiêu này không chỉ cải thiện chất lượng tiếng nói cảm nhận được, mà còn có thể tác động tích cực đến các tác vụ hạ nguồn tiếp theo như [nhận dạng người nói](../Recognition/Speaker_Recognition_and_Verification.md), [hệ thống chuyển văn bản thành tiếng nói (TTS) / tiếng nói thành văn bản (STT)](../Speech_Synthesis.md) hoặc dịch tự động.

![](attachments/bandwidth-extension/bluetooth-bwe-diagram.png)

Trong vài thập kỷ qua, xử lý tín hiệu số dựa trên học sâu đã chứng minh những kết quả vượt trội trong nhiều tác vụ như giảm nhiễu tiếng nói {cite:p}`braun2020denoising`, tổng hợp tiếng nói {cite:p}`tan2021speechsynthesis` hay chuyển đổi giọng nói {cite:p}`huan2020vcsurvey`. Tuy nhiên, một số giải pháp đòi hỏi lượng tính toán lớn hơn đáng kể so với các giải pháp truyền thống, khiến chúng không khả thi để xử lý trong thời gian thực. Một giải pháp thay thế để giảm thiểu vấn đề này là huấn luyện mạng nơ-ron xử lý đầu vào ở tần số lấy mẫu thấp hơn so với đầu ra mong muốn của một ứng dụng cụ thể, sau đó mở rộng băng thông một cách nhân tạo cho tín hiệu kết quả trước khi đưa tới đầu ra của hệ thống. Vì lượng tính toán của phần lớn các lớp mạng nơ-ron tăng tỷ lệ thuận với kích thước đầu vào, việc xử lý đầu vào có kích thước nhỏ hơn có thể cải thiện đáng kể hiệu suất tổng thể của hệ thống. Điều này đặc biệt quan trọng trên các thiết bị di động hoặc thiết bị biên (edge devices) nơi tài nguyên tính toán bị hạn chế. Hơn nữa, trong một số trường hợp, các thiết bị này là lựa chọn duy nhất vì lý do quyền riêng tư, vì việc triển khai trực tiếp trên thiết bị (on-device) giúp tránh truyền dữ liệu nhạy cảm lên máy chủ đám mây.

![](attachments/bandwidth-extension/postnet-bwe-diagram.png)

## Hướng tiếp cận hỗn hợp (Mixed approaches)
Do những hạn chế đã đề cập của cả phương pháp trên miền thời gian và miền tần số, một giải pháp thay thế là kết hợp ưu điểm của cả hai để giảm bớt các nhược điểm riêng lẻ {cite:p}`lin2021mixbwe`. Một cách thực hiện là kết nối tầng (cascade) hai mạng nơ-ron: Đầu tiên, một **mạng A** hoạt động trên miền tần số sẽ tạo ra một ước lượng dạng sóng thô có phổ biên độ $\hat{S}_h$ khớp với phổ biên độ của tín hiệu gốc $S_h$. Sau đó, **mạng B** thứ hai sẽ giúp tinh chỉnh các tín hiệu quá độ (transients) và giảm bớt các lỗi lệch pha. Vì phương pháp này bao gồm hai mạng nơ-ron kết nối tầng, kích thước tổng thể và yêu cầu tính toán của nó dự kiến sẽ tăng lên so với bất kỳ phương pháp mạng nơ-ron sâu đơn lẻ nào trước đó. Điều này có thể không quá quan trọng đối với các trường hợp sử dụng ngoại tuyến (offline) hoặc suy luận trên đám mây (cloud inferencing), nhưng có thể là yếu tố quyết định đối với các ứng dụng thời gian thực và suy luận trực tiếp trên thiết bị di động.

![](attachments/bandwidth-extension/mixed-bwe-diagram.png)

Các hướng tiếp cận khác sử dụng kết nối song song thay vì kết nối tầng hai mạng nơ-ron {cite:p}`lim2018mixbwe`.

## Mạng đối địch tạo sinh (Generative Adversarial Networks - GANs)
Mạng đối địch tạo sinh (GAN) đã chứng minh thành công vang dội trong nhiều tác vụ xử lý âm thanh {cite:p}`liu2020cpgan` {cite:p}`ferro2019cyclegan` {cite:p}`kong2020hifigan` bao gồm cả BWE {cite:p}`su2021bwegan` {cite:p}`liu2022bwegan` {cite:p}`li2020rtgan`. Ý tưởng chính là thay vì sử dụng một mạng nơ-ron đơn lẻ, GAN kết hợp ít nhất hai mạng đóng vai trò khác nhau đối lập nhau. Vai trò của **bộ tạo sinh $G$ (generator)** là tạo ra các mẫu ước lượng $\hat{s}_h$ sao cho càng gần với đầu ra mong muốn $s_h$ càng tốt. Trong khi đó, **bộ phân biệt $D$ (discriminator)** đóng vai trò là một bộ phân loại để phân biệt giữa các mẫu giả $\hat{s}_h$ (do bộ tạo sinh tạo ra) và các mẫu thật $s_h$. Cả hai mạng phải được huấn luyện xen kẽ để đạt được kết quả tốt nhất. Nếu **bộ phân biệt $D$** vượt trội hơn hẳn so với **bộ tạo sinh $G$**, bộ tạo sinh sẽ không thể học cách cải thiện việc tạo ra $\hat{s}_h$, dẫn đến hiện tượng được gọi là [vấn đề *triệt tiêu đạo hàm* (vanishing gradients)](https://en.wikipedia.org/wiki/Vanishing_gradient_problem).
Ngược lại, nếu **bộ tạo sinh $G$** vượt trội hơn hẳn so với **bộ phân biệt $D$**, nó có thể dẫn đến hiện tượng [*sụp đổ chế độ* (mode collapse)](https://wandb.ai/authors/DCGAN-ndb-test/reports/Measuring-Mode-Collapse-in-GANs--VmlldzoxNzg5MDk), nghĩa là **bộ tạo sinh $G$** chỉ tạo ra một số lượng rất ít các mẫu cụ thể mà nó biết chắc là sẽ đánh lừa được **bộ phân biệt $D$**. Đây là điều không mong muốn vì mục tiêu cuối cùng là tạo ra một **bộ tạo sinh $G$** có khả năng tổng quát hóa tốt cho bất kỳ tín hiệu tiếng nói đầu vào $s_l$ nào.

![](attachments/bandwidth-extension/gan-generator-training-diagram.png)

Khi **bộ tạo sinh $G$** được huấn luyện, các trọng số của **bộ phân biệt $D$** sẽ được đóng băng. Trước tiên, $G$ sẽ tạo ra một dự đoán và $D$ sẽ phân loại dự đoán này là thật hay giả. Sau đó, $G$ cập nhật các đạo hàm (gradients) của mình để dần dần tạo ra các dự đoán tốt hơn mà $D$ không thể phân biệt được với âm thanh băng thông đầy đủ (full-bandwidth) thực tế.

![](attachments/bandwidth-extension/gan-discriminator-training-diagram.png)

Sau đó, các trọng số của **bộ tạo sinh $G$** được đóng băng và nó được sử dụng để tạo ra các mẫu giả (tức là các mẫu đã được mở rộng băng thông một cách nhân tạo). Một hỗn hợp gồm các mẫu giả này và các mẫu thật (âm thanh gốc thực tế) sẽ được đưa vào **bộ phân biệt $D$** cùng với các nhãn tương ứng của chúng. Điều này giúp so sánh kết quả phân loại của $D$ trên cả hai loại mẫu so với nhãn thực tế.

Nếu hệ thống GAN hội tụ thành công, bước tiếp theo là ngắt kết nối **bộ phân biệt $D$** và chỉ sử dụng **bộ tạo sinh $G$** cho quá trình suy luận (inference). Điều này mang lại lợi thế lớn là loại bỏ hoàn toàn gánh nặng tính toán do **bộ phân biệt $D$** gây ra trong quá trình chạy thực tế. Ngoài ra, một số cấu hình có thể sử dụng nhiều bộ phân biệt cùng lúc, ví dụ: mỗi bộ phân biệt sử dụng STFT với kích thước FFT và kích thước bước nhảy (hop size) khác nhau. Nhìn chung, GAN vẫn là một lĩnh vực nghiên cứu rất tích cực và nhiều chi tiết triển khai được tinh chỉnh thông qua thử nghiệm thực nghiệm (thử và sai) cho từng trường hợp cụ thể.

# Đánh giá

Mặc dù có các chỉ số được sử dụng rộng rãi để đánh giá chất lượng tiếng nói tự động như [Đánh giá chất lượng tiếng nói theo cảm nhận (PESQ)](https://en.wikipedia.org/wiki/Perceptual_Evaluation_of_Speech_Quality) hoặc [Đánh giá chất lượng tiếng nói ảo theo mục tiêu người nghe (ViSQOL)](https://arxiv.org/abs/2004.09584), chúng thường yêu cầu đầu vào là băng hẹp (narrow band) hoặc băng rộng (wide band) để tính toán điểm số. Đối với bài toán BWE, việc làm này sẽ làm mất đi mục đích của thuật toán nếu mục tiêu là tạo ra tín hiệu có tần số lấy mẫu lớn hơn các mức đó. Vì lý do này, đánh giá chất lượng tự động cho tiếng nói băng thông đầy đủ (full-band) là một chủ đề nghiên cứu rất tích cực. Điểm đánh giá trung bình ý kiến (Mean Opinion Score - MOS) dựa trên đánh giá chủ quan vẫn tiếp tục là *tiêu chuẩn vàng (gold standard)* để lựa chọn ứng viên tốt nhất cho các tác vụ cụ thể, tuy nhiên quy trình này thường rất tốn kém và mất thời gian. Hơn nữa, nếu **bài báo A** công bố kết quả mà sau đó **bài báo B** tuyên bố vượt trội hơn với một điểm số MOS nhỉnh hơn một chút, điều này vẫn có thể gây tranh cãi nếu không có thêm thông tin chi tiết về quy trình đánh giá, số lượng người tham gia hoặc bất kỳ điều kiện bổ sung nào để đảm bảo sự so sánh công bằng giữa cả hai nghiên cứu.

Các chỉ số sau đây được báo cáo phổ biến nhất trong các nghiên cứu BWE tại thời điểm biên soạn tài liệu này:

## Khoảng cách phổ logarit (Log Spectral Distance - LSD)
Khoảng cách phổ logarit là một chỉ số trên miền tần số đo lường khoảng cách logarit giữa hai phổ. Vì nó dựa trên phổ biên độ, nó sẽ không xem xét đến độ chính xác của quá trình khôi phục pha:

$$\text{LSD}\left(S_h, \hat{S}_h\right)=\frac{1}{M}\sum\limits_{m=0}^{M-1}\sqrt{\frac{1}{K}\sum\limits_{k=0}^{K-1}\left(\text{log}_{10}\frac{\hat{S}_{h}\left(m, k\right)^2}{S_{h}\left(m, k\right)^2}\right)^2}$$

Trong đó $\hat{S}_h$ và $S_h$ lần lượt là phổ biên độ STFT của tín hiệu ước lượng và tín hiệu gốc thực tế (ground truth). Một số biến thể của chỉ số này sẽ chỉ tính toán trên phần phổ được khôi phục. Trong trường hợp đó, chỉ số này được gọi trong tài liệu là **LSD-HF** (LSD tần số cao). Ví dụ, nếu băng thông của tín hiệu được mở rộng từ 16 kHz lên 32 kHz, **LSD-HF** sẽ chỉ xem xét các bins FFT biểu diễn tần số trên 8 kHz.

## SNR
Tỷ số tín hiệu trên nhiễu (Signal-to-Noise Ratio - SNR) cung cấp một phép so sánh logarit trên miền thời gian, nhưng được tính giữa tín hiệu gốc $s_h$ và hiệu số giữa tín hiệu gốc $s_h$ với tín hiệu ước lượng $\hat{s}_h$:

$$\text{SNR}\left(\hat{s}_h, s_h\right) = 10\text{log}_{10}\frac{\sum_{n=0}^{N-1}s_h[n]^2}{\sum_{n=0}^{N-1}\left(\hat{s}_h[n]-s_h[n]\right)^2}$$

## SI-SDR
Tỷ số tín hiệu trên méo không đổi theo quy mô (Scale-Invariant Signal-to-Distortion Ratio - SI-SDR) {cite:p}`roux2018sisdr` cung cấp một phiên bản cải tiến của tỷ số tín hiệu trên méo (SDR) và tương tự như SNR, tương ứng với phép so sánh logarit trên miền thời gian giữa tín hiệu ước lượng $\hat{s}_h$ và tín hiệu gốc $s_h$:

$$\text{SI-SDR}=10\text{log}_{10}\left(\frac{\lVert e_{\text{target}}\rVert^2}{\lVert e_{\text{res}}\rVert^2}ight)$$

Trong đó:

$e_{\text{target}}=\frac{\hat{s}_h^T s_h}{\lVert s_h\rVert^2}s_h$ và $e_{\text{res}}=\hat{s}_h-e_{\text{target}}$

## Điểm đánh giá trung bình ý kiến (Mean Opinion Score - MOS)

Điểm đánh giá trung bình ý kiến (Mean Opinion Score - MOS) là một thước đo chủ quan thường được sử dụng trong kỹ thuật viễn thông (nhưng không giới hạn ở đó) để đại diện cho chất lượng tổng thể của một hệ thống. Trong trường hợp mở rộng băng thông, một người nghe đã qua đào tạo sẽ đánh giá chất lượng của một bản ghi âm đã được mở rộng băng thông nhân tạo. Chuẩn [ITU-T P.800.1](https://www.itu.int/rec/T-REC-P.800.1-201607-I/es) định nghĩa các cách sử dụng khác nhau của điểm số này. Đây là một giá trị nguyên duy nhất nằm trong khoảng từ 1 đến 5, trong đó chất lượng thấp nhất là 1 và chất lượng cảm nhận cao nhất là 5. Khi nhiều người nghe cùng đánh giá một thuật toán cụ thể, điểm MOS cuối cùng sẽ là trung bình cộng của tất cả các điểm số đánh giá và do đó có thể là một số thập phân khi lấy trung bình.

# Tài liệu tham khảo
```{bibliography}
:filter: docname in docnames
```